# Entrenar wake word "Hey ZIX" -> zix.onnx (openWakeWord)

Notebook basado en el pipeline oficial de openWakeWord (`notebooks/automatic_model_training.ipynb`),
ajustado para el asistente **ZIX**: la frase objetivo es **"Hey ZIX"** y el modelo final se llama **`zix.onnx`**.

**Flujo objetivo:** *"Hey ZIX" → wake word detectado → ZIX escucha → procesa el comando → responde → vuelve a esperar "Hey ZIX".*

**Requisitos:** Google Colab con runtime **GPU (T4)** (Runtime → Cambiar tipo de entorno → T4 GPU).
Tiempo estimado: **1-2 horas** (generación ~30-45 min, aumento ~10 min, entrenamiento ~30-60 min).

# 1. Instalación del entorno

Solo compatible con Linux (Colab lo es). Instala el fork de `piper-sample-generator` (TTS para las muestras sintéticas), el repo de openWakeWord completo (incluye `train.py`) y todas las dependencias de entrenamiento.

> Nota: si alguna instalación fallara en el Colab actual (los pins son de 2023), prueba cambiar el runtime a una versión de Python 3.10, o usa el notebook comunitario `alfiedennen/openwakeword-colab-2026` que fija versiones compatibles con 2026.

In [ ]:
## Environment setup (pipeline oficial de openWakeWord)

# piper-sample-generator (solo linux): TTS para generar ejemplos sinteticos
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# openwakeword completo (soporta entrenamiento)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

# Otras dependencias de entrenamiento
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Modelos de features compartidos (melspectrogram + embedding)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

In [ ]:
# Imports
import os
import sys
import uuid
from pathlib import Path
import numpy as np
import yaml
import datasets
import scipy
from tqdm import tqdm
import torch

# 2. Datos

El entrenamiento automático usa 4 fuentes de datos:
1. **Ejemplos sintéticos** de "Hey ZIX" y de frases adversariales (generados con Piper TTS en el paso de entrenamiento).
2. **Respuestas de impulso** (reverberación) del dataset de MIT.
3. **Audio de fondo/ruido** (AudioSet + Free Music Archive) para el aumento.
4. **Features precomputados negativos** (ACAV100M ~2000 h) y de validación (~11 h).

Estos datos se descargan en Colab (no hay que tocar nada de tu PC).

In [ ]:
# 2a. Respuestas de impulso de MIT (reverberacion)
output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset(
    "davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True
)
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(
        os.path.join(output_dir, name), 16000,
        (row['audio']['array'] * 32767).astype(np.int16)
    )

In [ ]:
# 2b. Audio de fondo: AudioSet (una parte del tar) + FMA (1 hora)
os.makedirs("audioset", exist_ok=True)
fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
os.makedirs(output_dir, exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict(
    {"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]}
)
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(
        os.path.join(output_dir, name), 16000,
        (row['audio']['array'] * 32767).astype(np.int16)
    )

output_dir = "./fma"
os.makedirs(output_dir, exist_ok=True)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
n_hours = 1
for i in tqdm(range(n_hours * 3600 // 30)):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(
        os.path.join(output_dir, name), 16000,
        (row['audio']['array'] * 32767).astype(np.int16)
    )

In [ ]:
# 2c. Features negativos precomputados (entrenamiento + validacion)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# 3. Configuración del entrenamiento

Configuración de **"Hey ZIX"** con `model_name: zix` (el `.onnx` final se llamará `zix.onnx`).

| Parámetro | Valor | Nota |
|---|---|---|
| `target_phrase` | `["hey zix"]` | Frase exacta a detectar |
| `model_name` | `zix` | Da nombre al archivo final `zix.onnx` |
| `n_samples` | 5000 | Muestras sintéticas positivas (10000+ = mejor) |
| `n_samples_val` | 2000 | Validación / early stopping |
| `steps` | 30000 | Pasos de entrenamiento (50000 = mejor) |
| `target_accuracy` | 0.7 | Métrica objetivo |
| `target_recall` | 0.5 | Métrica objetivo |
| `target_false_positives_per_hour` | 0.2 | Máximo de falsos positivos por hora |

In [ ]:
# Configuracion YAML para train.py (basada en examples/custom_model.yml)
config = {
    "model_name": "zix",
    "target_phrase": ["hey zix"],
    "custom_negative_phrases": [],
    "n_samples": 5000,
    "n_samples_val": 2000,
    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "piper_sample_generator_path": "./piper-sample-generator",
    "output_dir": "./zix_model",
    "rir_paths": ["./mit_rirs"],
    "background_paths": ["./audioset_16k", "./fma"],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": "./validation_set_features.npy",
    "augmentation_rounds": 1,
    "feature_data_files": {
        "ACAV100M_sample": "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
    },
    "batch_n_per_class": {
        "ACAV100M_sample": 1024,
        "adversarial_negative": 50,
        "positive": 50,
    },
    "model_type": "dnn",
    "layer_size": 32,
    "steps": 30000,
    "target_accuracy": 0.7,
    "target_recall": 0.5,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": 0.2,
}

with open("my_model.yaml", "w") as file:
    yaml.dump(config, file)

print("Configuracion guardada en my_model.yaml")
print(yaml.dump(config))

# 4. Entrenar

Tres pasos (puedes ejecutarlos uno a uno y ver el progreso):
1. **Generar clips sintéticos** de "Hey ZIX" (+ adversariales) con Piper.
2. **Aumentar** los clips con ruido/reverberación.
3. **Entrenar** el modelo (early stopping + checkpoint averaging + conversión automática a `.onnx`).

Si la generación fallara a mitad, basta con volver a ejecutar el paso 1: continúa desde donde quedó.

In [ ]:
# Paso 1: generar clips sinteticos (~30-45 min en T4)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

# 4b. Audios reales de "Hey ZIX" (opcional pero recomendado)

Mezcla tus grabaciones reales (distintas personas, distancias 0.5m/2m/5m, lugares, con/sin musica y ruido) con los clips sinteticos del paso 1.
Hazlo **despues** de `generate_clips` y **antes** de `augment_clips`.

1. Prepara en tu PC con `data/wakeword_real/README.md` + `prepare_real_clips.py` (WAV 16kHz mono).
2. Sube los WAV validados a `./hey_zix_real/` (panel Archivos -> Subir).
3. Ejecuta la celda de codigo: convierte a 16k/mono, copia 80% a `zix_model/positive/train/` y 20% a `zix_model/positive/validation/`.
4. Sigue con `augment_clips` + `train_model` normal.

Si no tienes audios aun, salta esta celda: el entrenamiento sintetico sigue funcionando.

In [ ]:
# Celda 4b: mezclar audios reales en zix_model/positive/ (ejecutar tras generate_clips)
import glob, os, random, shutil
from pathlib import Path
import numpy as np
import scipy.io.wavfile

src_dir = Path("./hey_zix_real")
pos_train = Path("./zix_model/positive/train")
pos_val = Path("./zix_model/positive/validation")
pos_train.mkdir(parents=True, exist_ok=True)
pos_val.mkdir(parents=True, exist_ok=True)

wavs = sorted(src_dir.glob("*.wav")) if src_dir.is_dir() else []
if not wavs:
    print("Sin WAV en ./hey_zix_real/. Subelos (panel Archivos) para usar 4b.")
    print("Se omite 4b: se entrena solo con sinteticos.")
else:
    random.seed(7)
    n_val = max(1 if len(wavs) >= 5 else 0, int(len(wavs) * 0.2))
    val_set = set(random.sample([str(w) for w in wavs], n_val)) if n_val else set()
    added_train, added_val, skipped = 0, 0, 0
    for wav in wavs:
        try:
            sr, data = scipy.io.wavfile.read(str(wav))
            if data.ndim > 1:
                data = data.mean(axis=1).astype(data.dtype)
            if sr != 16000:
                from scipy import signal
                n = int(len(data) * 16000 / sr)
                data = signal.resample(data.astype(np.float32), n)
                sr = 16000
                data = (np.clip(data, -1, 1) * 32767).astype(np.int16)
            dur = len(data) / 16000
            if not (0.5 <= dur <= 5.0):
                print(f"SKIP {wav.name}: duracion {dur:.2f}s")
                skipped += 1
                continue
            dest = pos_val / wav.name if str(wav) in val_set else pos_train / wav.name
            scipy.io.wavfile.write(str(dest), 16000, data.astype(np.int16) if data.dtype != np.int16 else data)
            if dest.parent == pos_val:
                added_val += 1
            else:
                added_train += 1
        except Exception as e:
            print(f"SKIP {wav.name}: {type(e).__name__}: {e}")
            skipped += 1
    print(f"Audios reales mezclados: train +{added_train}, val +{added_val}, omitidos {skipped}")
    print(f"Totales ahora: train={len(list(pos_train.glob('*.wav')))}, val={len(list(pos_val.glob('*.wav')))}")



In [ ]:
# Paso 2: aumentar los clips (~10 min)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Paso 3: entrenar (~30-60 min en T4). Genera zix.onnx (+ zix.tflite)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

# 5. Verificación y descarga

El modelo final está en `zix_model/zix.onnx`. Verificamos que:
- El archivo existe y tiene tamaño razonable.
- openWakeWord lo carga y produce scores con audio real del entrenamiento (positivo vs. fondo).

**Solo necesitas `zix.onnx`** para ZIX: los feature models compartidos (melspectrogram + embedding) ya están instalados en tu PC con el paquete `openwakeword`; openWakeWord no requiere metadatos adicionales junto al `.onnx`.

In [ ]:
import glob

onnx_files = glob.glob("./zix_model/*.onnx")
print("Archivos generados:")
for f in sorted(glob.glob("./zix_model/*")):
    print(f"  {f}  ({os.path.getsize(f) / 1024:.1f} KB)")
assert onnx_files, "No se genero ningun .onnx"

In [ ]:
# Carga el modelo entrenado y comprueba scores sobre clips reales
from openwakeword.model import Model

model = Model(wakeword_models=["./zix_model/zix.onnx"], inference_framework="onnx")

pos = sorted(glob.glob("./zix_model/positive/train/*.wav"))
neg = sorted(glob.glob("./zix_model/negative/train/*.wav"))
print(f"Clips positivos disponibles: {len(pos)}, negativos: {len(neg)}")

def score_first(wav):
    sr, data = scipy.io.wavfile.read(wav)
    if sr != 16000:
        return None
    out = model.predict(data)
    return max(out.values())

for wav in pos[:3]:
    print(f"POS  {os.path.basename(wav)}  score={score_first(wav):.3f}")
for wav in neg[:3]:
    print(f"NEG  {os.path.basename(wav)}  score={score_first(wav):.3f}")
print("\nEsperado: scores POS claramente > scores NEG.")

In [ ]:
# Descargar zix.onnx a tu PC (tambien puedes descargarlo desde el panel de archivos)
from google.colab import files
files.download("./zix_model/zix.onnx")

# Siguiente paso en ZIX (se hace tras tu aprobación)

1. Copiar `zix.onnx` a `models/wakeword/zix.onnx` en el proyecto.
2. En `.env`: `WAKE_WORD_MODEL=zix` (y calibrar `WAKE_WORD_THRESHOLD` si hace falta).
3. `hey_jarvis_v0.1.onnx` **se mantiene** como respaldo (volver con `WAKE_WORD_MODEL=hey_jarvis`).
4. Verificar: `python scripts/integration_test_voice.py` y prueba real de micrófono.

**No se reemplaza nada ni se toca la arquitectura de ZIX hasta tu aprobación.**